# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

This project uses the Refresh / Content Opportunity Scoring lane. FlyRank content teams may have more existing pages that deserve attention than reviewers can inspect in one work cycle. The question is: Can a supervised model rank likely declining content-refresh candidates more effectively than a transparent rules-based queue while using only public-safe, non-identifying content and search signals?
The decision supported is review prioritization. The output is a ranked queue, and the human action is to inspect the highest-priority candidates before deciding whether any editorial change is appropriate.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

I used the bundled anonymized 30,000-row teaching slice from the FlyRank ML Internship dataset. Each row represents an aggregated content item with content, search-performance, and engagement fields measured over a trailing 90-day window ending at export time. The decline comparison uses the last 30 days versus the previous 30 days.
The target is is_declining_label. It equals 1 when trend_direction == "down" and 0 otherwise. The dataset contains 16,262 positive rows, so the positive-label rate is 54.2%.
I excluded trend_direction and trend_pct because they define the target and would leak the answer. I also excluded titles, URLs, domains, keywords, client names, provider names, model names, raw queries, and identifiers as model features. Pseudonymous identifiers were used only to keep client groups separate during validation.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The pipeline creates 52 model features after preprocessing. Numeric features include search volume, competition, CPC, word count, character count, log impressions, log clicks, log sessions, AI sessions, days with impressions, days with sessions, content age, days since update, CTR, average position, engagement rate, scroll rate, and AI-traffic share. Categorical features include competition level, content type, main intent, age tier, freshness tier, word-count tier, impression tier, and position tier.
Missing numeric values were filled with zero, and missing categorical values were represented as unknown. I compared three supervised models—logistic regression, decision tree, and random forest—with a transparent hand-written rules baseline.
The random forest was selected using Precision@50 because the practical use case is prioritizing a small review queue. The validation design was a client-holdout split: 27,675 rows for training and 2,325 rows for testing. Client groups were kept separate between training and testing to reduce memorization across the same groups.
For leakage control, I excluded all target-derived trend fields from the features and did not use pseudonymous identifiers as predictors. The result is a predictive ranking aid, not a causal model.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

All methods were evaluated on the same client-holdout test split. The positive-label base rate was 54.2%.

| Method | ROC AUC | Average precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| Rule baseline | 0.627 | 0.468 | 0.240 | 0.189 | 0.274 |
| Logistic regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| Decision tree | 0.742 | 0.575 | 0.660 | 0.716 | 0.634 |
| Random forest | 0.747 | 0.610 | 0.680 | 0.741 | 0.638 |

The random forest measured Precision@50 of 0.680 versus 0.240 for the rules baseline. This is an observed absolute improvement of 0.440 on the same holdout split. It also measured ROC AUC of 0.747 versus 0.627 for the baseline.
The largest fitted feature importances were days_with_impressions, log_impressions_90d, avg_position, and content_age_days. These are predictive associations in this dataset, not proof that changing any one feature will cause performance to improve.
The artifacts include charts for model comparison, feature importance, action mix, confidence mix, reason codes, and trend distribution.

## 5. Limitations

*What this work cannot claim.*

This is a retrospective experiment on an anonymized teaching slice. It does not prove that refreshing content causes traffic or engagement to improve, predict Google’s ranking algorithm, or guarantee performance after deployment.
The target is a proxy for one observed definition of decline. The client-holdout split tests separation across client groups, but it is not a complete time-forward production simulation. The 30,000-row teaching slice is also smaller than the documented full warehouse release.
Performance may change across time windows, content types, traffic volumes, or client groups. The model should therefore be used only as a reviewer-prioritization aid. High-confidence candidates still require manual inspection, and any refresh intervention should be evaluated through a separate controlled experiment.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. Review the high-confidence candidates first, checking data quality and editorial context before making changes.
2. Prioritize pages with visible demand but weak CTR or engagement signals for human diagnosis rather than automatic refresh.
3. Use content age and recent-update recency as review context, not as automatic decision rules.
4. Treat the model score as a prioritization signal, not as a publishing, deletion, or rewriting instruction.
5. Evaluate any refresh intervention through a controlled before-and-after or holdout experiment.
6. Monitor performance across client groups, time windows, content types, and traffic-volume bands before broader operational use.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The analysis produces a ranked refresh queue containing model scores, confidence bands, recommended actions, and reason codes. It also produces a model-comparison table, feature-importance chart, action-mix chart, confidence-mix chart, reason-code chart, trend-distribution chart, generated Markdown report, and PDF summary.
The artifacts are descriptive and support reviewer prioritization. They do not automatically publish, delete, or rewrite content.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.